# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

/Users/pawel.pozorski/Desktop/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.utils.jupyter import display_shap_colors_df
from mllm_shap.shap import ComplementaryNeymanShapExplainer, Explainer
from mllm_shap.shap.normalizers import MinMaxNormalizer

Define LiquidAudio model (this call loads it up to the memory!).

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history

W0505 13:30:16.074000 1287 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create explainer that will make initial call and then explain it using shapley values using Neyman Formula. When `initial_samples` or `initial_fraction` are not provided, it will use default formula of `max(2, ceil(num_splits / (2 * n^2)))`.

In [6]:
explainer = Explainer(
    model=model,
    shap_explainer=ComplementaryNeymanShapExplainer(
        normalizer=MinMaxNormalizer(), num_samples=200
    ),
)

Create new chat instance and assign it messages. ComplementaryNeymanShapExplainer currently supports only `SystemRolesSetup.SYSTEM_ASSISTANT` mode. It requires at least one assistant turn to be present.

In [7]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,  # calculate shapley values for all roles
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.ASSISTANT)
chat.add_text("Be helpful and concise.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you? Where have been created?")
chat.end_turn()

# Usage

Let's calculate shapley values for current conversation.

Generation kwargs allows to customize model interference - here we limit it to 4 tokens and change text_temperature from default 0.0 to 0.2, text_top_k from default 1 to 3. 

In [8]:
generation_kwargs = {
    "max_new_tokens": 4,
    "model_config": ModelConfig(text_temperature=0.2, text_top_k=3),
}

result = explainer(
    chat=chat,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2026-05-05 13:30:20,930 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2026-05-05 13:30:26,401 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 9 (up to 511 additional calls)
2026-05-05 13:30:26,798 - mllm_shap.shap.neyman._base - INFO - Starting initial sampling step with 2 samples per entry in M


Neyman SHAP [stage 1/2]:   0%|          | 0/200 [00:00<?, ?it/s]

2026-05-05 13:30:40,380 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=92 cache_hits=0 cache_misses=92 skipped_filtered=0 model_elapsed_ms=12259.21
2026-05-05 13:30:40,380 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=46 yielded=92 skipped(full_or_empty)=0 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=13577.66
2026-05-05 13:30:44,056 - mllm_shap.shap.neyman._base - INFO - Starting Neyman allocation step with 56 remaining samples
2026-05-05 13:31:01,221 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=112 cache_hits=0 cache_misses=112 skipped_filtered=0 model_elapsed_ms=16604.26
2026-05-05 13:31:01,221 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=56 yielded=112 skipped(full_or_empty)=0 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=17164.08


Let's see final Shap values.

In [9]:
display_shap_colors_df(
    pd.DataFrame(
        list(
            zip(
                [chat.decode_text(token) for token in result.full_chat.input_tokens],
                result.full_chat.cache.normalized_values.tolist(),
            )
        ),
        columns=["Text", "Shapley Value"],
    )
)

,Text,Shapley Value
0,<|startoftext|>,nan
1,<|im_start|>,nan
2,assistant,nan
3,,nan
4,Be,nan
5,helpful,nan
6,and,nan
7,concise,nan
8,.,nan
9,<|im_end|>,nan


# Tests

Presence matrix:

In [10]:
explainer.shap_explainer._M

tensor([[ 2,  2,  7, 11, 10, 12, 15, 21, 22,  2],
        [ 2,  3,  4,  8,  6, 16, 18, 24, 21,  2],
        [ 2,  3,  6, 11, 10, 12, 15, 22, 21,  2],
        [ 2,  3,  6, 10, 11, 11, 16, 22, 21,  2],
        [ 2,  2,  3,  8,  9, 13, 18, 25, 22,  2],
        [ 2,  3,  7, 10, 10, 12, 16, 21, 21,  2],
        [ 2,  2,  6,  6,  8, 14, 20, 22, 22,  2],
        [ 2,  3,  6,  7, 15,  7, 19, 22, 21,  2],
        [ 2,  3, 11,  7,  9, 13, 19, 17, 21,  2]], device='mps:0',
       dtype=torch.int16)

CC matrices:

In [11]:
explainer.shap_explainer._C

tensor([[-0.5156, -0.1953, -0.1680,  0.3359,  0.0234,  0.6680,  1.5156,  2.6406,
          3.7969,  0.5156],
        [-0.5156, -0.6797, -0.5586, -0.3125, -0.0430,  0.6016,  0.8672,  2.2812,
          3.2969,  0.5156],
        [-0.5156, -0.3633,  0.2891,  0.3398,  0.4648,  1.1094,  1.5078,  3.1094,
          3.6094,  0.5156],
        [-0.5156, -0.3281, -0.1133, -0.2461,  0.1055,  0.7500,  0.9258,  2.7188,
          3.6719,  0.5156],
        [-0.5156, -0.3477, -0.5039,  0.2422, -0.6406,  0.0039,  1.4219,  2.3281,
          3.6094,  0.5156],
        [-0.5156, -0.6797, -1.3516, -1.0469, -0.0742,  0.5703,  0.1328,  1.4766,
          3.2969,  0.5156],
        [-0.5156, -0.3359, -0.7969, -0.8242, -0.5781,  0.0664,  0.3516,  2.0312,
          3.6406,  0.5156],
        [-0.5156, -0.4375, -0.8086, -0.9648, -1.0625, -0.4141,  0.2109,  2.0156,
          3.5312,  0.5156],
        [-0.5156, -0.6289, -1.6484, -1.0547, -0.7773, -0.1328,  0.1211,  1.1797,
          3.3906,  0.5156]], device='mps:0', dt

In [12]:
explainer.shap_explainer._BaseComplementaryNeymanShapExplainer__C_squared

tensor([[0.1328, 0.0201, 0.0503, 0.1738, 0.1699, 0.1670, 0.3281, 0.6445, 0.7109,
         0.1328],
        [0.1328, 0.1543, 0.1660, 0.1377, 0.1040, 0.2324, 0.3652, 0.5273, 0.5781,
         0.1328],
        [0.1328, 0.0439, 0.0791, 0.1660, 0.1621, 0.1738, 0.3379, 0.6133, 0.6875,
         0.1328],
        [0.1328, 0.0359, 0.1211, 0.1992, 0.1436, 0.1924, 0.3027, 0.5703, 0.6953,
         0.1328],
        [0.1328, 0.0659, 0.0908, 0.1084, 0.0688, 0.2676, 0.3945, 0.6016, 0.6641,
         0.1328],
        [0.1328, 0.1543, 0.2734, 0.2363, 0.1621, 0.1738, 0.2656, 0.4180, 0.5781,
         0.1328],
        [0.1328, 0.0608, 0.1309, 0.1289, 0.1250, 0.2109, 0.3730, 0.5625, 0.6719,
         0.1328],
        [0.1328, 0.0640, 0.1445, 0.1875, 0.1895, 0.1465, 0.3125, 0.5469, 0.6680,
         0.1328],
        [0.1328, 0.1328, 0.3301, 0.1699, 0.2207, 0.1157, 0.3340, 0.3633, 0.6016,
         0.1328]], device='mps:0', dtype=torch.bfloat16)